# Face Recognition with YOLOv8

Deep learning models learn hierarchical representations from raw data. **CNNs** apply learned filters spatially — detecting edges, textures, then high-level structures. ResNet, for example, stacks residual CNN blocks to classify images reliably. **EfficientNets** scale CNNs across depth, width, and resolution using a compound coefficient, achieving better accuracy per parameter — EfficientNet-B0 matches ResNet-50 at roughly 5× fewer parameters.

Here we fine-tune **YOLOv8**, which uses a CNN backbone, to solve a binary classification task: *owner* vs *not_owner*.

In [ ]:
!pip install ultralytics

## 1. Imports

In [ ]:
import os
import cv2
import uuid
import shutil
import random
from ultralytics import YOLO

## 2. Collect Images

A classifier needs labelled examples. We capture two classes:
- **owner** — photos of you
- **not_owner** — photos of anyone else, or varied backgrounds

YOLOv8 classification expects this folder structure:

```
data/dataset/
├── train/
│   ├── owner/
│   └── not_owner/
└── val/
    ├── owner/
    └── not_owner/
```

We collect raw images first, then split them automatically. Aim for at least 40–50 images per class.

Press **S** to save a frame, **Q** to quit early.

In [ ]:
RAW_PATH = os.path.join('data', 'raw')
CLASSES = ['owner', 'not_owner']
NUM_IMAGES = 40  # per class

for cls in CLASSES:
    os.makedirs(os.path.join(RAW_PATH, cls), exist_ok=True)

print("Directories ready:", os.listdir(RAW_PATH))

In [ ]:
# Capture OWNER images — sit in front of the camera
cap = cv2.VideoCapture(0)
cls = 'owner'
count = 0

while cap.isOpened() and count < NUM_IMAGES:
    ret, frame = cap.read()
    if not ret:
        break

    display = frame.copy()
    cv2.putText(display, f"{cls}  {count}/{NUM_IMAGES}  [S] save  [Q] quit",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.imshow('Capture', display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        path = os.path.join(RAW_PATH, cls, f"{cls}_{uuid.uuid1()}.jpg")
        cv2.imwrite(path, frame)
        count += 1
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Saved {count} images for '{cls}'")

In [ ]:
# Capture NOT_OWNER images — have someone else sit in front, or move away
cap = cv2.VideoCapture(0)
cls = 'not_owner'
count = 0

while cap.isOpened() and count < NUM_IMAGES:
    ret, frame = cap.read()
    if not ret:
        break

    display = frame.copy()
    cv2.putText(display, f"{cls}  {count}/{NUM_IMAGES}  [S] save  [Q] quit",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    cv2.imshow('Capture', display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        path = os.path.join(RAW_PATH, cls, f"{cls}_{uuid.uuid1()}.jpg")
        cv2.imwrite(path, frame)
        count += 1
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Saved {count} images for '{cls}'")

## 3. Build the Dataset

We split each class 80/20 into train and val. The model learns on the training split; the validation split measures how well it generalises to images it has never seen — which is what matters in practice.

In [ ]:
DATASET_PATH = os.path.join('data', 'dataset')
VAL_SPLIT = 0.2

for cls in CLASSES:
    src = os.path.join(RAW_PATH, cls)
    images = [f for f in os.listdir(src) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)

    cut = int(len(images) * (1 - VAL_SPLIT))
    splits = {'train': images[:cut], 'val': images[cut:]}

    for split, files in splits.items():
        dest = os.path.join(DATASET_PATH, split, cls)
        os.makedirs(dest, exist_ok=True)
        for f in files:
            shutil.copy(os.path.join(src, f), os.path.join(dest, f))

    print(f"{cls}: {cut} train / {len(images) - cut} val")

print("\nDataset ready at:", DATASET_PATH)

## 4. Fine-Tune YOLOv8

Rather than training from scratch, we load `yolov8n-cls.pt` — a nano classification model pre-trained on ImageNet. We then replace and retrain its head on our two-class dataset. This is **transfer learning**: the backbone already knows how to extract visual features; we only teach it what our specific face looks like. It converges quickly and works well even with small datasets.

In [ ]:
model = YOLO('yolov8n-cls.pt')

model.train(
    data=DATASET_PATH,
    epochs=30,
    imgsz=224,
    batch=16,
    name='face_recognition',
    patience=10,   # stop early if validation loss stops improving
)

print("Best weights saved to:", model.trainer.best)

## 5. Test on a Still Image

Before wiring the model into the live app, verify it predicts correctly on a held-out sample.

In [ ]:
best_model = YOLO(os.path.join('runs', 'classify', 'face_recognition', 'weights', 'best.pt'))

val_owner_dir = os.path.join(DATASET_PATH, 'val', 'owner')
sample = os.path.join(val_owner_dir, os.listdir(val_owner_dir)[0])

results = best_model(sample)
probs = results[0].probs
print("Predicted:", results[0].names[probs.top1])
print(f"Confidence: {probs.top1conf:.2%}")

## 6. Live Webcam Test

A final sanity check: run inference on your webcam feed in real time. Green = owner, red = not_owner. Press **Q** to exit.

In [ ]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = best_model(frame, verbose=False)
    probs = results[0].probs
    label = results[0].names[probs.top1]
    conf = probs.top1conf.item()

    color = (0, 200, 80) if label == 'owner' else (0, 60, 220)
    cv2.putText(frame, f"{label}  {conf:.0%}", (12, 42),
                cv2.FONT_HERSHEY_SIMPLEX, 1.1, color, 2)
    cv2.imshow('Face Recognition', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
cap = cv2.VideoCapture(0)
# Loop through labels
for label in labels:
    print('Collecting images for {}'.format(label))
    time.sleep(5)
    
    # Loop through image range
    for img_num in range(number_imgs):
        print('Collecting images for {}, image number {}'.format(label, img_num))
        
        # Webcam feed
        ret, frame = cap.read()
        
        # Naming out image path
        imgname = os.path.join(IMAGES_PATH, label+'.'+str(uuid.uuid1())+'.jpg')
        
        # Writes out image to file 
        cv2.imwrite(imgname, frame)
        
        # Render to the screen
        cv2.imshow('Image Collection', frame)
        
        # 2 second delay between captures
        time.sleep(2)
        
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
cap.release()
cv2.destroyAllWindows()

In [ ]:
for label in labels:
    print('Collecting images for {}'.format(label))
    for img_num in range(number_imgs):
        print('Collecting images for {}, image number {}'.format(label, img_num))
        imgname = os.path.join(IMAGES_PATH, label+'.'+str(uuid.uuid1())+'.jpg')
        print(imgname)   

In [ ]:
!pip install pyqt5 lxml --upgrade
!cd labelImg && pyrcc5 -o libs/resources.py resources.qrc

# 6. Load Custom Model

In [11]:
img = os.path.join('data', 'images', 'awake.c9a24d48-e1f6-11eb-bbef-5cf3709bbcc6.jpg')

In [13]:
results.print()

image 1/1: 480x640 1 awake
Speed: 16.0ms pre-process, 11.0ms inference, 2.0ms NMS per image at shape (1, 3, 480, 640)


In [15]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    
    # Make detections 
    results = model(frame)
    
    cv2.imshow('YOLO', np.squeeze(results.render()))
    
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()